# Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ctgan import CTGAN


## Dataset Overview

We load the dataset and inspect its structure to understand variable types and distributions.

# To be Deleted
Add all Dataset info here!!!!

## Test  Dataset Overview

For this test, we shall be using the Adult Income dataset. It contains demographic and employment-related attributes used to predict whether an individual's income exceeds $50K per year.

Sensitive attributes include:
- sex
- race

Target variable:
- income

In [ ]:
df = pd.read_csv("test_data/adult.csv")

df.head()

In [ ]:
df.info()

The dataset is preprocessed and included in the CTGAN library for demonstration purposes. It retains key features such as demographic attributes and income labels, making it suitable for fairness analysis.

## Categorical Variables

We explicitly define categorical variables required by CTGAN.

In [ ]:
categorical_columns = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
    "income",
]

# ORDER
- CTGAN
- TVAE
- FairGAN
- TabFairGAN

## Synthetic Data Generation using CTGAN

This notebook explores the use of CTGAN (Conditional Tabular GAN) for generating synthetic tabular data. The goal is to evaluate whether synthetic data preserves the statistical properties and fairness characteristics of the original dataset.

Synthetic data is increasingly used in machine learning to address issues such as data scarcity and privacy. However, since generative models learn from historical data, they may also reproduce or amplify existing biases.

This experiment investigates:
- The quality of synthetic data generated by CTGAN
- Whether bias present in the real dataset is preserved in the synthetic data

### Model Training

The CTGAN model learns the joint distribution of the dataset.

In [ ]:
import os

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

In [ ]:
model = CTGAN(epochs=200, verbose=True)

model.fit(df, discrete_columns=categorical_columns)

### Synthetic Data Generation

In [ ]:
ctgan_synthetic_data = model.sample(5000)

ctgan_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
ctgan_synthetic_data.to_csv("ctgan_synthetic.csv", index=False)

### Statistical Evaluation

In [ ]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in numerical_cols:
    plt.figure()
    sns.kdeplot(df[col], label="Real", fill=True)
    sns.kdeplot(ctgan_synthetic_data[col], label="Synthetic", fill=True)
    plt.title(col)
    plt.legend()
    plt.show()

### Fairness Analysis

We compare income outcomes across sensitive groups (e.g., sex).

In [ ]:
# Convert income to binary
df["income_binary"] = df["income"].apply(lambda x: 1 if ">50K" in x else 0)
ctgan_synthetic_data["income_binary"] = ctgan_synthetic_data["income"].apply(
    lambda x: 1 if ">50K" in x else 0
)

In [ ]:
# Statistical Parity
real_sp = df.groupby("sex")["income_binary"].mean()
syn_sp = ctgan_synthetic_data.groupby("sex")["income_binary"].mean()

print("Real:")
print(real_sp)

print("\nSynthetic:")
print(syn_sp)

In [ ]:
# Disparity Gap
real_gap = real_sp.max() - real_sp.min()
syn_gap = syn_sp.max() - syn_sp.min()

print("Real disparity:", real_gap)
print("Synthetic disparity:", syn_gap)

### Interpretation

If the disparity in synthetic data closely matches the real data, bias is preserved.

If the disparity increases, bias is amplified. If it decreases, bias is mitigated.

These findings highlight that synthetic data may replicate societal biases present in the original dataset.

# Synthetic Data Generation using TVAE

This notebook implements a TVAE (Tabular Variational Autoencoder) model to generate synthetic data.

The objective is to compare TVAE with CTGAN in terms of:
- Data quality
- Preservation of statistical properties
- Fairness across sensitive attributes

TVAE represents a probabilistic generative approach, in contrast to the adversarial training used in CTGAN.

# Synthetic Data Generation using FairGAN

This notebook implements a FairGAN model to generate synthetic data.

The objective is to compare FairGAN with non-fair models in terms of:
- Data quality
- Preservation of statistical properties
- Fairness across sensitive attributes

# Synthetic Data Generation using TabFairGAN

This notebook implements a TabFairGAN (Tabular Fair GAN) model to generate synthetic data.

The objective is to compare TabFairGAN with non-fair models in terms of:
- Data quality
- Preservation of statistical properties
- Fairness across sensitive attributes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tabfairgan import TFG

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

## Dataset

We use the Adult Income dataset which is our test data.

In [ ]:
df = pd.read_csv("test_data/adult.csv")

df.head()

## Data Preprocessing

We define categorical variables and clean the dataset.

In [ ]:
df = df.dropna()

categorical_columns = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
    "income",
]

## TabFairGAN Model Training

TabFairGAN differs from standard generative models such as CTGAN in that fairness constraints are incorporated directly into the training process.

Instead of using a separate fit function, the model is trained using a custom training loop.

### Fairness Constraint
To generate fair synthetic data, you need to pass a dictionary containing the following parameters:

- fair_epochs: Number of fair training epochs (integer).
- lamda: Lambda parameter controlling the trade-off between fairness and accuracy (float).
- S: Protected attribute (string, e.g., "sex").
- Y: Decision label (string, e.g., "income").
- S_under: Value representing the underprivileged group for the protected attribute (string, e.g., " Female").
- Y_desire: Desired value for the label (string, e.g., " >50K").

In [ ]:
# Define fairness configuration
fairness_config = {
    "fair_epochs": 50,
    "lamda": 0.5,
    "S": "sex",
    "Y": "income",
    "S_under": "Female",
    "Y_desire": ">50K",
}

# Initialize TabFairGAN with fairness constraints
model = TFG(df, epochs=200, batch_size=256, fairness_config=fairness_config)

# Train the model
model.train()

## Synthetic Data Generation

In [ ]:
tabfairgan_synthetic_data = model.generate_fake_df(num_rows=5000)

tabfairgan_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
tabfairgan_synthetic_data.to_csv("tabfairgan_synthetic.csv", index=False)

## Distribution Comparison

In [ ]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in numerical_cols:
    plt.figure()
    sns.kdeplot(df[col], label="Real", fill=True)
    sns.kdeplot(tabfairgan_synthetic_data[col], label="Synthetic", fill=True)
    plt.title(col)
    plt.legend()
    plt.show()

## Fairness Evaluation

In [ ]:
# Convert income to binary
df["income_binary"] = df["income"].apply(lambda x: 1 if ">50K" in x else 0)
tabfairgan_synthetic_data["income_binary"] = tabfairgan_synthetic_data[
    "income"
].apply(lambda x: 1 if ">50K" in x else 0)

In [ ]:
# Statistical Parity
real_sp = df.groupby("sex")["income_binary"].mean()
syn_sp = tabfairgan_synthetic_data.groupby("sex")["income_binary"].mean()

print("Real:")
print(real_sp)

print("\nSynthetic:")
print(syn_sp)

In [ ]:
# Disparity Gap
real_sp = df.groupby("sex")["income_binary"].mean()
syn_sp = tabfairgan_synthetic_data.groupby("sex")["income_binary"].mean()

print("Real:")
print(real_sp)

print("\nSynthetic:")
print(syn_sp)

## Downstream Classifier
To evaluate the utility of the generated data, standard machine learning classifiers will be trained using the synthetic datasets.

### Logistic Regression


#### Data Preprocessing

In [ ]:
# CLEAN DATA
def clean_dataframe(df):
    df = df.copy()

    # Remove leading/trailing spaces in strings
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

    return df


# Clean datasets
real_train_df = clean_dataframe(df)
synthetic_df = clean_dataframe(tabfairgan_synthetic_data)


real_train_df, real_test_df = train_test_split(
    real_train_df, test_size=0.2, random_state=42
)


def encode_target(df, target_col):
    df = df.copy()
    df[target_col] = df[target_col].astype(str)
    df[target_col] = df[target_col].str.strip()
    df[target_col] = df[target_col].str.replace(".", "", regex=False)
    df[target_col] = df[target_col].map({"<=50K": 0, ">50K": 1})
    return df


real_train_df = encode_target(real_train_df, "income")
real_test_df = encode_target(real_test_df, "income")
synthetic_df = encode_target(synthetic_df, "income")


#  BUILD PIPELINE
def build_pipeline(df, target_col):
    X = df.drop(columns=[target_col])

    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("num", "passthrough", numerical_cols),
        ]
    )

    pipeline = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(max_iter=1000)),
        ]
    )

    return pipeline


# FAIRNESS METRICS
def demographic_parity(y_pred, sensitive):
    groups = sensitive.unique()
    rates = [y_pred[sensitive == g].mean() for g in groups]
    return abs(rates[0] - rates[1])


def disparate_impact(y_pred, sensitive):
    groups = sensitive.unique()
    rate_0 = y_pred[sensitive == groups[0]].mean()
    rate_1 = y_pred[sensitive == groups[1]].mean()
    return min(rate_0 / (rate_1 + 1e-6), rate_1 / (rate_0 + 1e-6))


def equal_opportunity(y_true, y_pred, sensitive):
    groups = sensitive.unique()
    tprs = []

    for g in groups:
        mask = sensitive == g
        tp = ((y_pred == 1) & (y_true == 1) & mask).sum()
        fn = ((y_pred == 0) & (y_true == 1) & mask).sum()
        tprs.append(tp / (tp + fn + 1e-6))

    return abs(tprs[0] - tprs[1])


def equalized_odds(y_true, y_pred, sensitive):
    groups = sensitive.unique()
    tpr_list, fpr_list = [], []

    for g in groups:
        mask = sensitive == g

        tp = ((y_pred == 1) & (y_true == 1) & mask).sum()
        fn = ((y_pred == 0) & (y_true == 1) & mask).sum()
        fp = ((y_pred == 1) & (y_true == 0) & mask).sum()
        tn = ((y_pred == 0) & (y_true == 0) & mask).sum()

        tpr = tp / (tp + fn + 1e-6)
        fpr = fp / (fp + tn + 1e-6)

        tpr_list.append(tpr)
        fpr_list.append(fpr)

    return abs(tpr_list[0] - tpr_list[1]) + abs(fpr_list[0] - fpr_list[1])


# TRAIN + EVALUATE
def run_tabfairgan_evaluation(train_df, test_df, target_col, sensitive_col):

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]

    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col]
    sensitive = test_df[sensitive_col]

    model = build_pipeline(train_df, target_col)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "demographic_parity": demographic_parity(y_pred, sensitive),
        "disparate_impact": disparate_impact(y_pred, sensitive),
        "equal_opportunity": equal_opportunity(y_test, y_pred, sensitive),
        "equalized_odds": equalized_odds(y_test, y_pred, sensitive),
    }

    return results


# RUN IT
results = run_tabfairgan_evaluation(
    train_df=synthetic_df,  # train on synthetic
    test_df=real_test_df,  # test on real
    target_col="income",
    sensitive_col="sex",
)

print("TabFairGAN Results:")
print(results)

### Random Forest

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

## Dataset

We use the Adult Income dataset which is our test data.

In [ ]:
df = pd.read_csv("test_data/adult.csv")

df.head()

## Data Preprocessing

We define categorical variables and clean the dataset.

In [ ]:
df = df.replace("?", np.nan).dropna()
df.columns = [col.strip() for col in df.columns]

# Encode categorical variables
df_encoded = df.copy()

for col in df_encoded.columns:
    if df_encoded[col].dtype == "object":
        df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

### Define Sensitive and Target

In [ ]:
S = "sex"
Y = "income"

S_index = df_encoded.columns.get_loc(S)
Y_index = df_encoded.columns.get_loc(Y)

## FairGAN Model
### Model Architecture

FairGAN consists of:
- Generator
- Discriminator
- Fairness discriminator

In [ ]:
input_dim = df_encoded.shape[1]
noise_dim = 100


# Generator
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 128), nn.ReLU(), nn.Linear(128, input_dim)
        )

    def forward(self, z):
        return self.model(z)


# Real/Fake Discriminator (D1)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)


# Fairness Discriminator (D2)
class FairnessDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)

### Model Initialization

In [ ]:
G = Generator()
D1 = Discriminator()
D2 = FairnessDiscriminator()

criterion = nn.BCELoss()

g_optimizer = optim.Adam(G.parameters(), lr=0.001)
d1_optimizer = optim.Adam(D1.parameters(), lr=0.001)
d2_optimizer = optim.Adam(D2.parameters(), lr=0.001)

data = torch.tensor(df_encoded.values, dtype=torch.float32)

epochs = 200
lambda_fair = 0.05

## FairGAN Model Training

In [ ]:
for epoch in range(epochs):

    # Train D1 (real vs fake)
    real_labels = torch.ones(len(data), 1)
    fake_labels = torch.zeros(len(data), 1)

    z = torch.randn(len(data), noise_dim)
    fake_data = G(z)

    d1_real = D1(data)
    d1_fake = D1(fake_data.detach())

    d1_loss = criterion(d1_real, real_labels) + criterion(d1_fake, fake_labels)

    d1_optimizer.zero_grad()
    d1_loss.backward()
    d1_optimizer.step()

    # Train D2 (predict S)
    S_real = data[:, S_index].unsqueeze(1)

    d2_pred = D2(data)
    d2_loss = criterion(d2_pred, S_real)

    d2_optimizer.zero_grad()
    d2_loss.backward()
    d2_optimizer.step()

    # Train Generator
    z = torch.randn(len(data), noise_dim)
    fake_data = G(z)

    # GAN loss (fool D1)
    g_loss = criterion(D1(fake_data), real_labels)

    # FAIRNESS loss (Xu-style)
    S_fake_pred = D2(fake_data)

    # push D2 to be uncertain (0.5)
    target_uncertain = torch.full_like(S_fake_pred, 0.5)
    fair_loss = criterion(S_fake_pred, target_uncertain)

    # TOTAL LOSS
    total_g_loss = g_loss - lambda_fair * fair_loss

    g_optimizer.zero_grad()
    total_g_loss.backward()
    g_optimizer.step()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch} | D1: {d1_loss.item():.4f} | D2: {d2_loss.item():.4f} | G: {g_loss.item():.4f} | Fair: {fair_loss.item():.4f}"
        )

## Synthetic Data Generation

In [ ]:
z = torch.randn(len(data), noise_dim)
synthetic = G(z).detach().numpy()

fairgan_synthetic_data = pd.DataFrame(synthetic, columns=df_encoded.columns)

fairgan_synthetic_data.head()

Export synthetic data for analysis

In [ ]:
fairgan_synthetic_data.to_csv("fairgan_synthetic.csv", index=False)

## Distribution Comparison

In [ ]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in numerical_cols:
    if col in fairgan_synthetic_data.columns:
        plt.figure()

        sns.kdeplot(df[col], label="Real", fill=True)
        sns.kdeplot(fairgan_synthetic_data[col], label="FairGAN", fill=True)

        plt.title(f"Distribution Comparison: {col}")
        plt.legend()
        plt.show()

## Fairness Evaluation

In [ ]:
# Convert income to binary
df["income_binary"] = df["income"].apply(lambda x: 1 if ">50K" in x else 0)
fairgan_synthetic_data["income_binary"] = (
    fairgan_synthetic_data["income"] > 0.5
).astype(int)

df["sex_binary"] = df["sex"].apply(lambda x: 1 if "Male" in x else 0)
fairgan_synthetic_data["sex_binary"] = (
    fairgan_synthetic_data["sex"] > 0.5
).astype(int)

In [ ]:
# Statistical Parity
def statistical_parity(df, sensitive_col, outcome_col):
    group_rates = df.groupby(sensitive_col)[outcome_col].mean()
    return group_rates.max() - group_rates.min(), group_rates


real_spd, real_rates = statistical_parity(df, "sex_binary", "income_binary")
fake_spd, fake_rates = statistical_parity(
    fairgan_synthetic_data, "sex_binary", "income_binary"
)

print("REAL SPD:", real_spd)
print(real_rates)

print("\nFAKE SPD (FairGAN):", fake_spd)
print(fake_rates)

In [ ]:
print("Fake distribution:")
print(fairgan_synthetic_data["income_binary"].value_counts(normalize=True))

In [ ]:
# Disparity Gap
def disparate_impact(df, sensitive_col, outcome_col):
    group_rates = df.groupby(sensitive_col)[outcome_col].mean()
    return group_rates.min() / group_rates.max()


real_di = disparate_impact(df, "sex_binary", "income_binary")
fake_di = disparate_impact(
    fairgan_synthetic_data, "sex_binary", "income_binary"
)

print("REAL DI:", real_di)
print("FAKE DI (FairGAN):", fake_di)

In [ ]:
labels = ["Real", "FairGAN"]
spd_values = [real_spd, fake_spd]
di_values = [real_di, fake_di]

plt.figure()
plt.bar(labels, spd_values)
plt.title("Statistical Parity Difference")
plt.show()

plt.figure()
plt.bar(labels, di_values)
plt.title("Disparate Impact")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ctgan import TVAE

## Dataset

We use the Adult Income dataset which is our test data.

In [ ]:
df = pd.read_csv("test_data/adult.csv")

df.head()

## Data Preprocessing

We define categorical variables and clean the dataset.

In [ ]:
df = df.dropna()

categorical_columns = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
    "income",
]

## TVAE Model Training

TVAE learns a latent representation of the data and reconstructs it through a decoder network.

Unlike CTGAN, TVAE optimizes a probabilistic objective rather than using adversarial training.

In [ ]:
model = TVAE(epochs=200, verbose=True)

model.fit(df, discrete_columns=categorical_columns)

### Synthetic Data Generation

In [ ]:
tvae_synthetic_data = model.sample(5000)

tvae_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
tvae_synthetic_data.to_csv("tvae_synthetic.csv", index=False)

### Distribution Comparison

In [ ]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in numerical_cols:
    plt.figure()
    sns.kdeplot(df[col], label="Real", fill=True)
    sns.kdeplot(tvae_synthetic_data[col], label="Synthetic", fill=True)
    plt.title(col)
    plt.legend()
    plt.show()

### Fairness Evaluation

In [ ]:
# Convert income to binary
df["income_binary"] = df["income"].apply(lambda x: 1 if ">50K" in x else 0)
tvae_synthetic_data["income_binary"] = tvae_synthetic_data["income"].apply(
    lambda x: 1 if ">50K" in x else 0
)

In [ ]:
# Statistical Parity
real_sp = df.groupby("sex")["income_binary"].mean()
syn_sp = tvae_synthetic_data.groupby("sex")["income_binary"].mean()

print("Real:")
print(real_sp)

print("\nSynthetic:")
print(syn_sp)

In [ ]:
# Disparity Gap
real_sp = df.groupby("sex")["income_binary"].mean()
syn_sp = tvae_synthetic_data.groupby("sex")["income_binary"].mean()

print("Real:")
print(real_sp)

print("\nSynthetic:")
print(syn_sp)